# Sonata + Cosine Masking on Colab
## Opencode Generated Notebook — 5 Cells, 3 Scripts

1. Run **CELL 1** → restart when prompted → continue with CELL 2-5.

In [ ]:
# CELL 1 — Install Condacolab + Create Environment
# After this, Colab prompts restart. Click Restart, then continue.
# IMPORTANT: condacolab.install() must run INLINE (needs IPython kernel for restart).
%cd /content
!rm -rf /content/voxel_group_classifier
!git clone --depth 1 https://github.com/yuhang-wang-xjtu/voxel_group_classifier.git /content/voxel_group_classifier
%cd /content/voxel_group_classifier
!pip install condacolab
import condacolab
condacolab.install()

### ▲ ▲ ▲ ▲ ▲
**After restart: go to Runtime → Restart and run all → Continue from here.**
### ▼ ▼ ▼ ▼ ▼

In [ ]:
# CELL 2 -- Install Dependencies (5-10 min)
%cd /content/voxel_group_classifier
!apt-get install -y -qq libsparsehash-dev 2>&1 | tail -1
# Create conda env from environment_colab.yml (skip if exists)
!conda env create -f environment_colab.yml 2>&1 | tail -5
# Verify torch works in conda env
!conda run -n pointcept-torch2.5.0-cu12.4 python -c 'import torch; print(f"torch={torch.__version__} cuda_avail={torch.cuda.is_available()}")'
# Install CUDA extensions (need nvcc from system CUDA in PATH)
!CUDA_HOME=/usr/local/cuda PATH=/usr/local/cuda/bin:$PATH conda run -n pointcept-torch2.5.0-cu12.4 pip install ./libs/pointops
!CUDA_HOME=/usr/local/cuda PATH=/usr/local/cuda/bin:$PATH conda run -n pointcept-torch2.5.0-cu12.4 pip install ./libs/pointops2
!CUDA_HOME=/usr/local/cuda PATH=/usr/local/cuda/bin:$PATH conda run -n pointcept-torch2.5.0-cu12.4 pip install ./libs/pointgroup_ops
print('Done.')

In [ ]:
# CELL 2.5 - Install flash-attention (for Sonata enable_flash=True)
# Required by Point Transformer V3 backbone. Skip if this fails (training still works, just slower).
!conda run -n pointcept-torch2.5.0-cu12.4 pip install flash-attn --no-build-isolation

In [ ]:
# CELL 3 - Download S3DIS + Convert (3 min, ~1.7 GB)
%cd /content/voxel_group_classifier
!conda run -n pointcept-torch2.5.0-cu12.4 python scripts/colab_data.py

In [ ]:
# CELL 4 — Train Sonata + Cosine Masking
# Arguments: --mode=[cosine|random|vgc]  --epochs=10
%cd /content/voxel_group_classifier
!conda run -n pointcept-torch2.5.0-cu12.4 python scripts/colab_train.py --mode=cosine --epochs=10

In [ ]:
# CELL 5 — Monitor
%load_ext tensorboard
%tensorboard --logdir /content/voxel_group_classifier/exp --port 6006